#Capstone: Predict Flight Delays at Albuquerque International Sunport (ABQ)

##Problem Definition

Supervised

##Data Collection

In [18]:
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn                 import datasets
from sklearn.metrics         import mean_squared_error
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from IPython.display         import display
from sklearn                 import tree

from google.colab import userdata
import os

In [2]:
url = 'https://www.transtats.bts.gov/OT_Delay/ot_delaycause1_DL.aspx?8n4=9ur4r%20Brn4z106u%20or69rr0%20FHFHE%20NaQ%20FHGEK%20n0q%20Nv42146%20=%MENOd%ME'
urllib.request.urlretrieve(url, 'delay_causes.zip')

('delay_causes.zip', <http.client.HTTPMessage at 0x7c71cef143b0>)

In [3]:
#Unzip file
!unzip -o delay_causes.zip

Archive:  delay_causes.zip
  inflating: Airline_Delay_Cause.csv  
  inflating: Download_Column_Definitions.xlsx  


In [4]:
#Verify
!ls -la

total 216
drwxr-xr-x 1 root root   4096 Jul 27 17:04 .
drwxr-xr-x 1 root root   4096 Jul 27 17:00 ..
-rw-r--r-- 1 root root 153717 Jul 27 13:04 Airline_Delay_Cause.csv
drwxr-xr-x 4 root root   4096 Jun  4 13:32 .config
-rw-r--r-- 1 root root  32911 Jul 27 17:04 delay_causes.zip
-rw-r--r-- 1 root root  11215 Apr 24  2024 Download_Column_Definitions.xlsx
drwxr-xr-x 1 root root   4096 Jun  4 13:32 sample_data


In [5]:
#Check headers and file output
!head -n 1 Airline_Delay_Cause.csv | tr ',' '\n' | cat -n

     1	year
     2	month
     3	carrier
     4	carrier_name
     5	airport
     6	airport_name
     7	arr_flights
     8	arr_del15
     9	carrier_ct
    10	weather_ct
    11	nas_ct
    12	security_ct
    13	late_aircraft_ct
    14	arr_cancelled
    15	arr_diverted
    16	arr_delay
    17	carrier_delay
    18	weather_delay
    19	nas_delay
    20	security_delay
    21	late_aircraft_delay


In [6]:
df = pd.read_csv('Airline_Delay_Cause.csv')
df

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2026,5,MQ,Envoy Air,ABQ,"Albuquerque, NM: Albuquerque International Sun...",43.0,10.0,4.00,0.94,...,0.0,1.87,0.0,1.0,701.0,290.0,116.0,118.0,0.0,177.0
1,2026,5,OO,SkyWest Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun...",470.0,69.0,43.05,6.10,...,0.0,8.80,1.0,2.0,4027.0,2525.0,360.0,498.0,0.0,644.0
2,2026,5,QX,Horizon Air,ABQ,"Albuquerque, NM: Albuquerque International Sun...",31.0,1.0,1.00,0.00,...,0.0,0.00,0.0,0.0,46.0,46.0,0.0,0.0,0.0,0.0
3,2026,5,UA,United Air Lines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun...",232.0,53.0,21.40,0.89,...,0.0,21.47,2.0,2.0,2784.0,996.0,47.0,365.0,0.0,1376.0
4,2026,5,AA,American Airlines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun...",240.0,76.0,28.22,3.45,...,0.0,34.83,9.0,0.0,5019.0,1663.0,452.0,437.0,0.0,2467.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879,2020,1,OO,SkyWest Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun...",285.0,37.0,23.36,1.89,...,0.0,3.00,2.0,0.0,1791.0,902.0,89.0,425.0,0.0,375.0
880,2020,1,UA,United Air Lines Network,ABQ,"Albuquerque, NM: Albuquerque International Sun...",74.0,12.0,1.96,0.00,...,0.0,6.60,0.0,0.0,586.0,55.0,0.0,218.0,0.0,313.0
881,2020,1,WN,Southwest Airlines,ABQ,"Albuquerque, NM: Albuquerque International Sun...",881.0,64.0,28.64,0.00,...,0.0,25.56,6.0,2.0,2838.0,1346.0,0.0,326.0,0.0,1166.0
882,2020,1,YV,Mesa Airlines Inc.,ABQ,"Albuquerque, NM: Albuquerque International Sun...",149.0,22.0,7.53,0.90,...,0.0,10.61,1.0,0.0,1837.0,228.0,131.0,139.0,0.0,1339.0


##Data Cleaning

In [7]:
df.shape

(884, 21)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 884 entries, 0 to 883
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 884 non-null    int64  
 1   month                884 non-null    int64  
 2   carrier              884 non-null    object 
 3   carrier_name         884 non-null    object 
 4   airport              884 non-null    object 
 5   airport_name         884 non-null    object 
 6   arr_flights          883 non-null    float64
 7   arr_del15            882 non-null    float64
 8   carrier_ct           883 non-null    float64
 9   weather_ct           883 non-null    float64
 10  nas_ct               883 non-null    float64
 11  security_ct          883 non-null    float64
 12  late_aircraft_ct     883 non-null    float64
 13  arr_cancelled        883 non-null    float64
 14  arr_diverted         883 non-null    float64
 15  arr_delay            883 non-null    flo

**Columns**

Categorical (*Nominal*):
- carrier               object
- carrier_name          object
- airport               object
- airport_name          object

Categorical (*Ordinal*):
- month                 convert from int64   to category

Numerical (*Discrete*):
- year                  convert from int64   to int16
- arr_flights           convert from float64 to int32
- **arr_del15           convert from float64 to int32**    (Regression or Classification)
- arr_cancelled         convert from float64 to int32
- arr_diverted          convert from float64 to int32

Numerical (*Continuous*):
- carrier_ct            convert from float64 to float32
- weather_ct            convert from float64 to float32
- nas_ct                convert from float64 to float32
- security_ct           convert from float64 to float32
- late_aircraft_ct      convert from float64 to float32
- **arr_delay           convert from float64 to float32**  (Regression)
- carrier_delay         convert from float64 to float32
- weather_delay         convert from float64 to float32
- nas_delay             convert from float64 to float32
- security_delay        convert from float64 to float32
- late_aircraft_delay   convert from float64 to float32

Potential Target Variables




###Target

In [9]:
target = 'arr_delay'
df[target].head

<bound method NDFrame.head of 0       701.0
1      4027.0
2        46.0
3      2784.0
4      5019.0
        ...  
879    1791.0
880     586.0
881    2838.0
882    1837.0
883       0.0
Name: arr_delay, Length: 884, dtype: float64>

###Unique IDs

In [10]:
identifier_cols = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_cols.append(col)

print(identifier_cols)
#No unique IDs

[]


###Rows

In [11]:
#Rows with nulls
df.isnull().any(axis = 1).sum()

np.int64(2)

In [12]:
#Missing rows
df.isnull().sum().sort_values() *1000

,0
year,0
month,0
carrier,0
carrier_name,0
airport,0
airport_name,0
arr_flights,1000
carrier_ct,1000
late_aircraft_ct,1000
weather_ct,1000


In [13]:
#Duplicate rows
df.duplicated().sum()

np.int64(0)

###Columns

In [14]:
cols = list(df.drop([target], axis=1).columns.sort_values())
cols

['airport',
 'airport_name',
 'arr_cancelled',
 'arr_del15',
 'arr_diverted',
 'arr_flights',
 'carrier',
 'carrier_ct',
 'carrier_delay',
 'carrier_name',
 'late_aircraft_ct',
 'late_aircraft_delay',
 'month',
 'nas_ct',
 'nas_delay',
 'security_ct',
 'security_delay',
 'weather_ct',
 'weather_delay',
 'year']

In [15]:
df.dtypes.value_counts()

,count
float64,15
object,4
int64,2


####Categorical

####Numerical

###Make a copy

#Save as a Parquet

In [34]:
parquet_file = 'delay_causes.parquet'
parquet_file

'delay_causes.parquet'

In [35]:
df.to_parquet(parquet_file, index=False)

In [36]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 49k Jul 27 17:21 delay_causes.parquet


In [37]:
df2 = pd.read_parquet(parquet_file)
df2.shape

(884, 21)

In [38]:
os.environ["HF_TOKEN"] = userdata.get('hf_cs_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_ld ... iLe'

In [50]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account

'stephanie465337'

In [51]:
hf_org = 'ddds-Capstone'
os.environ['HF_ORG'] = hf_org
hf_org

'ddds-Capstone'

In [49]:
hf_repo = "OT_Delay_Causes"
os.environ["HF_REPO"] = hf_repo
hf_repo

'OT_Delay_Causes'

In [46]:
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Capstone Token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [52]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ORG}/${HF_REPO} \
  delay_causes.parquet

In [53]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/ddds-Capstone/OT_Delay_Causes/commit/b89cd2fa78bd22ae3fd1bd370fa9405d6f1614e5



In [54]:
hf_url = f"https://huggingface.co/datasets/{hf_org}/{hf_repo}/resolve/main/delay_causes.parquet"
hf_url

'https://huggingface.co/datasets/ddds-Capstone/OT_Delay_Causes/resolve/main/delay_causes.parquet'

In [55]:
df3 = pd.read_parquet(hf_url)
df3.shape

(884, 21)

In [56]:
df3.iloc[:,:5].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 884 entries, 0 to 883
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          884 non-null    int64 
 1   month         884 non-null    int64 
 2   carrier       884 non-null    object
 3   carrier_name  884 non-null    object
 4   airport       884 non-null    object
dtypes: int64(2), object(3)
memory usage: 34.7+ KB
